In [1]:
%pip install pandas scikit-learn catboost xgboost seaborn matplotlib joblib sentence-transformers faiss-cpu torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from catboost import CatBoostRegressor, Pool
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import pickle
import faiss
import torch
import warnings
import os
RANDOM_STATE = 42
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.utils._auth")

C:\Users\Alpha\anaconda3\envs\testProjects\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("car_prices_extended_eda.csv", parse_dates=False)
df.columns

Index(['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state',
       'condition', 'odometer', 'color', 'interior', 'seller', 'mmr',
       'sellingprice', 'saledate', 'price_diff', 'car_age', 'mileage_per_year',
       'sale_year', 'sale_month', 'sale_day', 'seller_category',
       'condition_category', 'sale_day_of_week', 'sale_day_name', 'sale_hour',
       'is_weekend', 'sale_month_name', 'price_gap_pct', 'gap_category',
       'is_luxury'],
      dtype='object')

In [4]:
drop_cols = ["vin", "saledate", "price_diff"]
df = df.drop(columns=drop_cols)

In [5]:
# Log-skewed cols (as before)
skewed_cols = ["odometer", "car_age"]
df[skewed_cols] = np.log1p(df[skewed_cols])

In [6]:
# X and y
# Transform the target to log-scale for training (log(1+x))
y = np.log1p(df["sellingprice"])
X = df.drop(columns=["sellingprice"])

In [7]:
# Auto-detect cat_features (run this once)
cat_features = X.select_dtypes(include=["object"]).columns.tolist()
print("cat_features:", cat_features)

cat_features: ['make', 'model', 'trim', 'body', 'transmission', 'state', 'color', 'interior', 'seller', 'seller_category', 'condition_category', 'sale_day_name', 'sale_month_name', 'gap_category']


In [8]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Train with Pool (efficient)
train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features) # Create test pool for eval_set
model = CatBoostRegressor(
    iterations=600,
    depth=6,
    learning_rate=0.07,
    random_seed=42,
    early_stopping_rounds=50,
    task_type='CPU',
    use_best_model=True,
    verbose=100
)
model.fit(train_pool, eval_set=test_pool) # Pass eval_set to fit method

CatBoostError: bad allocation

In [ ]:
y_train_pred_log = model.predict(X_train)
y_test_pred_log = model.predict(X_test)

# Actuals (already log-transformed in Cell 20)
y_train_actual_log = y.loc[X_train.index]
y_test_actual_log = y.loc[X_test.index]

# 1. Back-Transform predictions and actuals to dollar scale
y_train_actual_dollars = np.expm1(y_train_actual_log)
y_test_actual_dollars = np.expm1(y_test_actual_log)
y_train_pred_dollars = np.expm1(y_train_pred_log)
y_test_pred_dollars = np.expm1(y_test_pred_log)

# 2. Calculate metrics (MAE, RMSE) on the DOLLAR SCALE
train_mae = mean_absolute_error(y_train_actual_dollars, y_train_pred_dollars)
test_mae = mean_absolute_error(y_test_actual_dollars, y_test_pred_dollars)
train_rmse = np.sqrt(mean_squared_error(y_train_actual_dollars, y_train_pred_dollars))
test_rmse = np.sqrt(mean_squared_error(y_test_actual_dollars, y_test_pred_dollars))

print(f"Train MAE: ${train_mae:.0f} | Test MAE: ${test_mae:.0f} | Gap: {((train_mae - test_mae) / test_mae * 100):+.1f}%")
print(f"Train RMSE: ${train_rmse:.0f} | Test RMSE: ${test_rmse:.0f} | Gap: {((train_rmse - test_rmse) / test_rmse * 100):+.1f}%")

# Threshold: If test/train ratio <0.85 (test 15% worse), overfitting likely
if test_rmse / train_rmse < 0.85:
    print("⚠️ Potential Overfitting: Test underperforms train significantly.")
else:
    print("✅ Good Fit: Metrics are comparable.")

In [11]:
# Bin prices
bins = [0, 2000, 5000, 7000,9000,12000,14000,16000,18000,20000,  np.inf]  # Low (<15k), Medium (15-25k), High (>25k)
labels = ['<2000','2000-5000', '5001-7000', '7001-9000', '9001-12000','12001-14000', '14001-16000', '16001-18000', '18001-20000', '>20001']

# Predict
y_pred_continuous = model.predict(X_test)
y_pred_binned = pd.cut(y_pred_continuous, bins=bins, labels=labels, include_lowest=True)

# Bin y_test for comparison
y_test_binned = pd.cut(y_test_continuous, bins=bins, labels=labels, include_lowest=True)

# Convert binned series to string type to handle potential NaN values and ensure consistent types for confusion_matrix
y_test_binned_str = y_test_binned.astype(str)
y_pred_binned_str = y_pred_binned.astype(str)

# Confusion Matrix
cm = confusion_matrix(y_test_binned_str, y_pred_binned_str, labels=labels)
print("Confusion Matrix:\n", cm)

# Report
print("\nClassification Report:\n", classification_report(y_test_binned_str, y_pred_binned_str))

# Plot (as in notebook)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix for Binned Car Prices')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

NameError: name 'y_test_continuous' is not defined

In [ ]:
# Evaluate
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"\nBaseline MAE on test set: ${mae:.2f}")
print(f"\nRMSE on test set: ${rmse:.2f}")